In [ ]:
# Chapter 3 Test: Dictionaries and Sets

# Answer each question in the code cell(s) below it. Some ask for code, some for written explanation, some for predicting output. **Do not run the prediction questions before answering** — the point is to reason through them first.


## Q1: What Makes a Valid Dict Key?

a) What requirement must an object meet to be usable as a dict key? Name the two dunder methods involved.

b) Explain why a `list` cannot be a dict key but a `tuple` can — then describe a tuple that *also* cannot be a dict key. What determines the difference?

c) How does this connect to the concept of container vs flat sequences from Chapter 2?

In [ ]:
# a) An object must be hashable and immutable in order to be a dict key.
# b) list is mutable so can't be dict key and a tuple is immutable so it can be a dict but when a tuple contains
# mutable object e.g. a list then a tuple can't be a dict key. So a tuple containing immutable objects can be a
# dict key.
# c) Not sure how container vs flat sequence is relevant here; I mean even flat sequence that is mutable can't be
# used as a key and a container sequence like a tuple can be used as a dict key when itself is immutable and 
# it only contains immutable objects; so not sure this question is "correct" or I am missing something fundamental here!


## Q2: Dict Comprehension

Given a list of `(city, country)` tuples:

```python
cities = [('Oslo', 'Norway'), ('Helsinki', 'Finland'), ('Oslo', 'Norway'), ('Tokyo', 'Japan'), ('Helsinki', 'Finland')]
```

Using a dict comprehension, create a dict mapping each **country** to a **list of unique cities** in that country. Do it in as few statements as possible.

In [9]:
# Your answer here
from collections import defaultdict
cities = [('Oslo', 'Norway'), ('Helsinki', 'Finland'), ('Oslo', 'Norway'), ('London', 'UK'),
          ('Tokyo', 'Japan'), ('Helsinki', 'Finland'), ('London', 'UK'), ('Manchester', 'UK')]
country_with_cities = defaultdict(set)
for (city, country) in cities:
    country_with_cities[country].add(city) 
{country: list(cities) for (country, cities) in country_with_cities.items()}

{country: list({city for city, c in cities if c == country}) for country in {c for _, c in cities}}

{'UK': ['Manchester', 'London'],
 'Norway': ['Oslo'],
 'Japan': ['Tokyo'],
 'Finland': ['Helsinki']}

## Q3: The Missing Key Problem — Predict & Explain

**Do not run first.** Predict what happens with each approach to handling a missing key:

```python
# Approach A
inventory = {}
for item in ['apple', 'banana', 'apple', 'cherry', 'banana', 'apple']:
    inventory[item] = inventory.get(item, 0) + 1

# Approach B
from collections import defaultdict
inventory2 = defaultdict(int)
for item in ['apple', 'banana', 'apple', 'cherry', 'banana', 'apple']:
    inventory2[item] += 1

# Approach C
inventory3 = {}
for item in ['apple', 'banana', 'apple', 'cherry', 'banana', 'apple']:
    inventory3.setdefault(item, 0)
    inventory3[item] += 1
```

a) Do all three produce the same result?

b) Now the deeper question: `defaultdict` calls its `default_factory` when a key is missing via `__missing__`. If you access `inventory2['grape']` (never inserted), what happens to `inventory2`? Why does this matter?

c) In which scenario is `setdefault` preferable over `defaultdict`?

In [ ]:
# a) yes they all produce the same result.
# b) it will insert grape with a count of 0 as that's what int() (the default factory returns).
# c) When the default value is not what a common default factory would return, for instance, if we want to set the default count for
# each fruit to be 10 for some reason; then setdefault is preferable over defaultdict.


## Q4: `__missing__` and the Lookup Chain

a) When you do `d[key]` on a dict subclass and the key is not found, describe the lookup chain. Which method is tried before `KeyError` is raised?

b) Does `d.get(key)` trigger `__missing__`? Does `key in d`? Explain why.

c) Write a `CaseInsensitiveDict` class that inherits from `dict` and implements `__missing__` so that looking up `'Python'` finds the value stored under `'python'`. Only `__getitem__` (i.e., `d[key]`) needs to be case-insensitive.

In [ ]:
# a) `d[key]` will first call __getitem__() and if not found then it will try to call __missing__() before raising the KeyError.
# b) I think they both do as they are both equivalent to d[key]
# c)
class CaseInsensitiveDict(dict):
    def __missing__(self, key):
        return self[key.lower()] 


## Q5: Immutable Mappings — Connecting to Chapter 2

a) What does `types.MappingProxyType` do? What happens if you try to assign a key on the proxy?

b) If you hold a `MappingProxyType` wrapping a regular dict, and someone mutates the underlying dict, does the proxy reflect the change? Why?

c) How does this relate to the tuple-with-mutable-items lesson from Chapter 2? (Think about what "immutable view" really means vs "immutable data".)

In [ ]:
# a) MappingProxyType is a readonly view of some mapping type and it will raise an error when assigning a key as it's readonly.
# b) Yes the proxy does reflect the change because they share the underlying map memory.
# c) This is similar to tuple being immutable but when it contains mutable items then that mutuable items can be updated thus the tuple
# also gets updated; so tuples containing mutable items are not strictly immutable! And while MappingProxyType itself is immutable
# its value can still be changed if the underlying dict it wraps is mutated.


## Q6: Merging Dicts — `|` vs `|=` vs `{**d1, **d2}`

a) What is the difference between `d1 | d2` and `d1 |= d2`?

b) When two dicts have overlapping keys, which dict's values win in `d1 | d2`?

c) Can `d1 |= some_iterable` accept something other than a dict on the right side? If so, what form must it take? How does this connect to how `dict()` constructor itself accepts iterables?

In [ ]:
# a) d1 | d2 will merge two dicts into a new dict whereas d1 |= d2 will merge d2 into d1.
# b) d2 will win as it's the latter.
# c) Yes it can be something other than a dict on the right side and as long as the elements in the iterable is hashable e.g. can be 
# used as a dict key. I guess d1 |= some_iterable is equivalent to create a new dict e.g. `d2 = dict(some_iterable)` and then d1 |= d2.


## Q7: `ChainMap` — Predict the Behavior

**Do not run first.**

```python
from collections import ChainMap

defaults = {'theme': 'dark', 'lang': 'en', 'timeout': 30}
user_prefs = {'theme': 'light'}
session = {'timeout': 5}

config = ChainMap(session, user_prefs, defaults)
```

a) What is `config['theme']`? What is `config['lang']`? What is `config['timeout']`?

b) If you do `config['font'] = 'mono'`, which of the three underlying dicts gets modified?

c) How is this different from merging the three dicts with `defaults | user_prefs | session`? Think about what happens if `defaults` is later changed — does `config` see it?

In [ ]:
# a) config['theme'] = 'light', config['lang'] = 'en', config['timeout'] = 5
# b) all three
# c) it's different from merging three dicts because direct merging is an one-off operation whereas ChainMap will link the new dict
# with underlying dict to maintain their relationship.


## Q8: Dict Views Are Live — Predict the Output

**Do not run first.**

```python
d = {'x': 1, 'y': 2, 'z': 3}
keys = d.keys()
vals = d.values()

d['w'] = 4
del d['x']

print(len(keys))
print('w' in keys)
print(list(vals))
```

a) What gets printed?

b) Why do views behave this way? What are they, structurally?

c) `d.keys()` supports set operations like `&` and `|`. Does `d.values()` support them? Why or why not?

In [ ]:
# a) 3, True, [2, 3, 4]
# b) Views are readonly
# c) `d.Values()` doesn't support set operations because unlike d.keys() which are hashable d.Values() are not.

## Q9: Set Theory in Practice

Given:
```python
backend_devs = {'alice', 'bob', 'carol', 'dave'}
frontend_devs = {'carol', 'dave', 'eve', 'frank'}
on_leave = {'bob', 'eve'}
```

Using **only set operations** (no loops, no comprehensions), find:

a) Developers who work on both backend and frontend.  
b) Developers who are available (not on leave) and work on at least one of backend or frontend.  
c) Developers who work exclusively on backend (not frontend) and are available.

In [ ]:
backend_devs = {'alice', 'bob', 'carol', 'dave'}
frontend_devs = {'carol', 'dave', 'eve', 'frank'}
on_leave = {'bob', 'eve'}

# a)
full_stack = backend_devs & frontend_devs
available_devs = (backend_devs | frontend_devs) - on_leave
available_backend_devs = (backend_devs - frontend_devs) - on_leave


## Q10: Sets and Hashability — The Shared Foundation

a) Write a set comprehension that, given a list of words, produces the set of word lengths present:

```python
words = ['distributed', 'system', 'cache', 'node', 'distributed', 'cache']
```

b) Why can a set only contain hashable elements? What underlying data structure does a set share with dict keys?

c) Given this shared foundation: if you need to test whether any of 1000 items exist in a collection of 1,000,000 elements, why is a set massively faster than a list? What's the algorithmic difference?

In [ ]:
# Your answer here
# a)
words = ['distributed', 'system', 'cache', 'node', 'distributed', 'cache']
word_lengths = {len(w) for w in words}

# b) The underlying data structure for both dict keys and set is hash table which requires elements to be hashable.
# c) Using set achieves O(n) whereas using a list requires O(n^2) for checking existance.

## Q11: The Hash Contract — Why Equal Objects Must Hash Equal

**Predict what happens. Do not run first.**

```python
class BrokenPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
    # no __hash__ defined

p1 = BrokenPoint(3, 4)
p2 = BrokenPoint(3, 4)
```

a) Can you use `p1` as a dict key in Python 3? Why or why not?

b) Now imagine `__hash__` was poorly defined to return `id(self)`. Both `p1 == p2` is `True` but `hash(p1) != hash(p2)`. If you do `d = {p1: 'found'}` then `d[p2]`, what goes wrong?

c) State the hash contract in one sentence.

In [ ]:
# Your answer here
# a) p1 can't be used a dict key because it's not hashable as __hash__ is not implemented.
# b) It raises KeyNotFound error because p2 doesn't exist in d.
# c) Two hashable objects that are equal must have the same hash.

## Q12: Pattern Matching with Mappings

Write a function `handle_event(event)` that uses `match/case` to handle a dict-based event:

- If it has `{'type': 'click', 'x': x, 'y': y}` → return `f'Click at ({x}, {y})'`
- If it has `{'type': 'keypress', 'key': k}` → return `f'Key: {k}'`
- If it has `{'type': 'scroll', **rest}` → return `f'Scroll with extra: {rest}'`
- Otherwise → return `'Unknown event'`

After writing it, answer: can the event dict have *extra* keys beyond what the pattern specifies and still match? Why is this a deliberate design choice for mappings but not for sequences?

In [ ]:
# Your answer here
def handle_event(event):
    match event:
        case {'type': 'click', 'x': x, 'y': y}:
            return f'Click at ({x}, {y})'
        case {'type': 'keypress', 'key': k}:
            return f'Key: {k}'
        case {'type': 'scroll', **rest}:
            return f'Scroll with extra: {rest}'
        case _:
            return 'Unkonwn event'

# because of the 3rd case with **rest, it will match any extra keys.
# Not sure why it's allowed on mappings but not on sequences; honestly I thought on sequences it works too but guess I was wrong!

## Q13: `UserDict` vs Subclassing `dict` — The Pitfall

a) Why does the book recommend subclassing `collections.UserDict` instead of `dict` directly when building a custom mapping?

b) Specifically: if you subclass `dict` and override `__setitem__`, will `d.update(...)` call your overridden `__setitem__`? Why is this problematic?

c) How does `UserDict` solve this? Where does its data actually live?

In [ ]:
# Your answer here
# a) I think it's because dict is used to store attributes in a custom type. Honestly that's just guess.
# b) No because of the reason in a)
# c) UserDict solves this by using a class attribute of type dict to store its data.

## Q14: `dict` Ordering vs `OrderedDict` — Why Both Exist

a) Since Python 3.7, `dict` preserves insertion order. So why does `collections.OrderedDict` still exist? Name at least two behaviors `OrderedDict` supports that regular `dict` does not.

b) **Predict the output. Do not run first.**

```python
d1 = {'a': 1, 'b': 2, 'c': 3}
d2 = {'c': 3, 'b': 2, 'a': 1}
print(d1 == d2)

from collections import OrderedDict
od1 = OrderedDict([('a', 1), ('b', 2), ('c', 3)])
od2 = OrderedDict([('c', 3), ('b', 2), ('a', 1)])
print(od1 == od2)
```

What gets printed and why?

In [ ]:
# a) have no idea!!!
# b) True, False; because for two OrderedDict to be equal they have to match on the order as well where a normal dict equality
# doesn't enforce this though it does preserve the inserting order of the elements.



In [10]:
d = defaultdict(list)
d['x'].append(1)
d

defaultdict(list, {'x': [1]})

In [13]:
d.get('y') == None

True

In [12]:
'z' in d

False

In [15]:
d2 = dict()
d.get('y') == None

True

In [1]:
450*0.86 - 40

347.0